# PDF → HTML (Paddle table detection)

Самостоятельный ноутбук — **не требует** `pdf_to_html_smart.ipynb`.

Пайплайн:
1. **Paddle PP-Structure** — bbox таблиц на растре страницы
2. **Расширение bbox** — padding + боковые подписи
3. **pdfplumber lines** на crop; fallback **text**
4. `pdf_html_pipeline.process_table_detected`
5. HTML: таблицы + свободный текст

```bash
pip install -e ".[dev,paddle]"
```

In [5]:
from pathlib import Path

ROOT = Path(".").resolve()
SAMPLES = ROOT / "samples"
OUTPUT = ROOT / "samples_html_detection"

In [6]:
import time
import warnings

import pdfplumber

from pdf_html_pipeline import (
    assemble_page_html,
    build_cells,
    classify_rows,
    is_prose_table,
    process_table_detected,
    prose_grid_to_sections,
    render_text_block_html,
    wrap_html_document,
)
from pdf_paddle_detection import PaddleTableDetector, find_tables_paddle
from pdf_table_engine import _suppress_scan_noise, table_looks_like_prose

warnings.filterwarnings("ignore")
_suppress_scan_noise()

In [7]:
def build_page_body_with_detection(
    page,
    page_num: int,
    pdf_path: str,
    detector: PaddleTableDetector | None = None,
) -> str:
    det = detector or PaddleTableDetector()
    detected = find_tables_paddle(page, pdf_path, page_num, det)

    processed: list[tuple] = []
    prose_sections: list[tuple[tuple, str]] = []

    for item in detected:
        table = item.table
        if table_looks_like_prose(page, table):
            grid = build_cells(page, table)
            kinds = classify_rows(grid)
            as_prose = True
        else:
            grid, kinds = process_table_detected(
                page, table, side_labels_in_bbox=True
            )
            as_prose = is_prose_table(grid)

        if as_prose:
            bbox = table.bbox
            for section_html in prose_grid_to_sections(
                grid, render_text_block_html, page.width
            ):
                prose_sections.append(((bbox[1], bbox[0]), section_html))
        processed.append((grid, kinds, table, as_prose))

    return assemble_page_html(page, processed, prose_sections)

In [8]:
def export_samples_to_html_detection(
    samples_dir: Path = SAMPLES,
    output_dir: Path = OUTPUT,
    detector: PaddleTableDetector | None = None,
) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    pdfs = sorted(samples_dir.glob("*.pdf"))
    det = detector or PaddleTableDetector()

    total_pages = 0
    total_seconds = 0.0
    exported: list[str] = []

    for pdf_path in pdfs:
        t0 = time.perf_counter()
        sections: list[str] = []
        with pdfplumber.open(pdf_path) as pdf:
            for pnum, page in enumerate(pdf.pages, start=1):
                total_pages += 1
                body = build_page_body_with_detection(
                    page, pnum, str(pdf_path), detector=det
                )
                sections.append(
                    f'<section class="page" data-page="{pnum}">\n{body}\n</section>'
                )
        doc = wrap_html_document("\n".join(sections), title=pdf_path.stem)
        out_path = output_dir / f"{pdf_path.stem}.html"
        out_path.write_text(doc, encoding="utf-8")
        exported.append(out_path.name)
        print(f"{pdf_path.name} -> {out_path.name} ({time.perf_counter() - t0:.1f}s)")
        total_seconds += time.perf_counter() - t0

    return {
        "pdf_count": len(pdfs),
        "total_pages": total_pages,
        "total_seconds": round(total_seconds, 2),
        "avg_seconds_per_page": round(total_seconds / max(total_pages, 1), 3),
        "exported_files": exported,
        "output_dir": str(output_dir),
    }

In [5]:
# Диагностика одной страницы
PDF = SAMPLES / "2508007948.pdf"
PAGE_NUM = 12

detector = PaddleTableDetector(show_log=False)

with pdfplumber.open(PDF) as pdf:
    page = pdf.pages[PAGE_NUM - 1]
    found = find_tables_paddle(page, PDF, PAGE_NUM, detector)
    print(f"Paddle tables: {len(found)}")
    for i, item in enumerate(found, 1):
        print(f"  {i}. raw={tuple(round(x, 1) for x in item.raw_bbox)}")
        print(f"     expanded={tuple(round(x, 1) for x in item.bbox)} "
              f"extract={item.extract_source}")
        grid, kinds = process_table_detected(page, item.table)
        n_cols = max((len(r) for r in grid), default=0)
        print(f"     grid={len(grid)} rows, cols={n_cols}")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/private/.paddlex/official_models/PP-DocLayout_plus-L`.


Paddle tables: 2
  1. raw=(84.8, 74.9, 543.3, 424.2)
     expanded=(80.8, 70.9, 547.3, 428.2) extract=lines
     grid=7 rows, cols=5
  2. raw=(89.1, 595.5, 536.8, 731.9)
     expanded=(85.1, 591.5, 540.8, 735.9) extract=text
     grid=0 rows, cols=0


In [ ]:
stats = export_samples_to_html_detection()
print(
    f"\nЭкспорт: {stats['pdf_count']} PDF, {stats['total_pages']} стр. "
    f"-> {stats['output_dir']}/\n"
    f"Среднее: {stats['avg_seconds_per_page']} с/стр."
)

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/private/.paddlex/official_models/PP-DocLayout_plus-L`.


2508007948-10.pdf -> 2508007948-10.html (32.3s)
2508007948.pdf -> 2508007948.html (267.3s)
2703000015-104.pdf -> 2703000015-104.html (25.7s)


In [2]:
from __future__ import annotations

import time
from pathlib import Path

import pdfplumber

from pdf_paddle_detection import PaddleTableDetector


def export_pdf_to_html(
    pdf_path: str | Path,
    html_path: str | Path,
    *,
    detector: PaddleTableDetector | None = None,
    page_from: int = 1,
    page_to: int | None = None,
) -> dict:
    """
    Конвертирует PDF в HTML и сохраняет по заданному пути.

    Args:
        pdf_path: путь к PDF
        html_path: путь к выходному .html (директория создаётся автоматически)
        detector: переиспользовать один PaddleTableDetector для пакетной обработки
        page_from: первая страница (1-based)
        page_to: последняя страница включительно; None = до конца

    Returns:
        dict со статистикой прогона
    """
    pdf_path = Path(pdf_path).resolve()
    html_path = Path(html_path).resolve()
    html_path.parent.mkdir(parents=True, exist_ok=True)

    if not pdf_path.is_file():
        raise FileNotFoundError(pdf_path)

    det = detector or PaddleTableDetector(show_log=False)
    t0 = time.perf_counter()
    sections: list[str] = []

    with pdfplumber.open(pdf_path) as pdf:
        last = page_to if page_to is not None else len(pdf.pages)
        if page_from < 1 or last > len(pdf.pages) or page_from > last:
            raise ValueError(
                f"Неверный диапазон страниц: {page_from}..{last} "
                f"(в PDF {len(pdf.pages)} стр.)"
            )

        for pnum in range(page_from, last + 1):
            page = pdf.pages[pnum - 1]
            body = build_page_body_with_detection(
                page, pnum, str(pdf_path), detector=det
            )
            sections.append(
                f'<section class="page" data-page="{pnum}">\n{body}\n</section>'
            )

    doc = wrap_html_document("\n".join(sections), title=pdf_path.stem)
    html_path.write_text(doc, encoding="utf-8")

    elapsed = time.perf_counter() - t0
    page_count = len(sections)
    return {
        "pdf_path": str(pdf_path),
        "html_path": str(html_path),
        "page_count": page_count,
        "seconds": round(elapsed, 2),
        "avg_seconds_per_page": round(elapsed / max(page_count, 1), 3),
    }

In [9]:
stats = export_pdf_to_html(
    "/Users/private/Desktop/pdf-to-html/samples/2508007948.pdf",
    "/Users/private/Desktop/pdf-to-html/samples_html/2508007948.html",
    detector=PaddleTableDetector(show_log=False),
)
print(stats)

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/private/.paddlex/official_models/PP-DocLayout_plus-L`.


{'pdf_path': '/Users/private/Desktop/pdf-to-html/samples/2508007948.pdf', 'html_path': '/Users/private/Desktop/pdf-to-html/samples_html/2508007948.html', 'page_count': 28, 'seconds': 476.91, 'avg_seconds_per_page': 17.032}


In [10]:
from IPython.display import display, HTML

display(HTML('/Users/private/Desktop/pdf-to-html/samples_html/2508007948.html'))